# Fuzzy Control with `fuzzy_torch.envs`

In notebook 01 we built fuzzy systems and trained them on data. Here we put them to
work as **controllers** and **watch them swing, drive and fly** - every simulation in
this notebook is shown as a video right below the cell that runs it.

| case study | plant | what the controller does |
|---|---|---|
| **Overhead crane** | `CranePlant` | move a trolley 5 m without letting the payload swing |
| **Fuzzy Grand Prix** | `RacingEnv` (differential-drive kart) | finish a lap as fast as possible without leaving the track |
| **Quadrotor** | `QuadrotorPlant` | climb to and hover at a target altitude |

All plants are **batched, differentiable torch models** behind the standard
**Gymnasium** API (`reset` / `step`), so the same plant serves classical controllers,
fuzzy controllers, RL agents and gradient-based tuning.

## 0. Setup

The cell below works on **Google Colab** (it installs the library with its simulation
extras and fetches the example files) and **locally** in a checkout of the repository.

In [ ]:
import os, sys, pathlib, importlib

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    os.environ.setdefault('MUJOCO_GL', 'egl')      # headless 3-D rendering on Colab GPUs
    !pip -q install "torch-fis[viz] @ git+https://github.com/mdrs-thiago/fuzzy_torch.git"
    !git clone -q --depth 1 https://github.com/mdrs-thiago/fuzzy_torch.git /content/fuzzy_torch_repo
    REPO = pathlib.Path('/content/fuzzy_torch_repo')
else:
    # This notebook lives in fuzzy_torch/examples/notebooks/.
    REPO = pathlib.Path.cwd().resolve().parents[1]
    try:
        importlib.import_module('fuzzy_torch')
    except ModuleNotFoundError:
        sys.path.insert(0, str(REPO.parent))   # flat package: put its parent on sys.path

import json
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
import fuzzy_torch as ft

torch.manual_seed(0)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('fuzzy_torch', ft.__version__, '| repo at', REPO)

In [ ]:
from fuzzy_torch import FuzzyVariable, FuzzyRule, FuzzyModule, TriangularMF, TrapezoidalMF
from fuzzy_torch.envs import (
    CranePlant, QuadrotorPlant, ControlMode, Scenario,
    step_response, disturbance_rejection, step_response_metrics, disturbance_metrics,
    FuzzyControlEnv, ErrorFeatures, RacingEnv, make_track, record, compare,
)
from fuzzy_torch.envs.controllers import crane_baseline, FuzzyRaceDriver, ConfigRaceDriver
from fuzzy_torch.envs.core.render import mujoco_available

### How simulations are shown

`record(env, policy)` runs one episode and returns a **video** that plays inline when it
is the last line of a cell. It also keeps every observation, action, `info` dict and the
raw plant state (`video.states`), so the same run feeds the plots.

- `RENDER_MODE = 'rgb_array'` captures frames for inline videos (works everywhere).
- `RENDER_MODE = 'human'` instead opens a **live window** - only when running on your own
  computer. The rest of the notebook stays exactly the same.

The renderer is **MuJoCo** (3-D) when an OpenGL context is available, otherwise
**pygame** (2-D, top-down). On Colab, MuJoCo needs a GPU runtime
(*Runtime → Change runtime type*); on a CPU runtime you get the pygame views.

In [ ]:
RENDER_MODE = 'rgb_array'      # 'human' -> live window (local only)

try:
    ok, why = mujoco_available(CranePlant())
except Exception as exc:       # a broken GL setup must not stop the notebook
    ok, why = False, repr(exc)
BACKEND = 'mujoco' if ok else 'pygame'
print('renderer:', '3-D MuJoCo' if ok else f'2-D pygame  ({why})')

## 1. Anatomy of an environment

Every benchmark has the same layers:

```
 Plant                -> physics: state, derivatives(state, u), bounds  (pure torch)
 Scenario             -> episode length, setpoint, scripted disturbances
 FuzzyControlEnv      -> Gymnasium wrapper: reset(), step(action), integrator, rendering
 ErrorFeatures        -> optional wrapper: raw state -> [error, error_dot, ...]
 your controller      -> obs -> action
```

In [ ]:
crane = CranePlant()
print('state      :', crane.state_names)
print('tracked    :', crane.tracked_states)
print('actions    :', crane.action_names, '| bounds', [b.tolist() for b in crane.action_bounds()])
print('default dt :', crane.default_dt, 's')

env = ErrorFeatures(FuzzyControlEnv(CranePlant(), scenario=step_response(500, setpoint=[5.0, 0.0])))
obs, info = env.reset(seed=0)
print('\nfeatures   :', env.feature_names)
print('first obs  :', obs)
obs, reward, terminated, truncated, info = env.step([10.0])   # push with 10 N
print('after 10 N :', obs)
print('info keys  :', list(info))

## 2. Overhead crane

A trolley carries a payload on a cable. Moving the trolley makes the payload swing;
the controller must reach the 5 m setpoint **and** damp the sway.

First, the provided **state-feedback baseline**,
$F = K \cdot [e_x, \dot e_x, e_\theta, \dot e_\theta]$. Watch the payload swing:

In [ ]:
STEP = step_response(1500, setpoint=[5.0, 0.0])      # 15 s: move from 0 m to 5 m

def crane_env(scenario=STEP):
    return ErrorFeatures(FuzzyControlEnv(CranePlant(), scenario=scenario,
                                         render_mode=RENDER_MODE, render_backend=BACKEND))

baseline = crane_baseline()
base_video = record(crane_env(), baseline, every=5)
base_video

The same run, as plots. `crane_log` pulls the trolley position and sway out of
`video.states`, and the commanded force out of the `info` dicts.

In [ ]:
def crane_log(video):
    infos = video.infos
    return {
        't':         np.array([i['time'] for i in infos]),
        'x':         video.states[:, 0],
        'theta':     video.states[:, 2],
        'force':     np.array([i['control'][0] for i in infos]),
        'target':    np.array([i['setpoint'][0] for i in infos]),
        'disturbed': np.array([i['disturbance'] is not None for i in infos]),
    }


def plot_crane(logs, title):
    fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
    for name, log in logs.items():
        axes[0].plot(log['t'], log['x'], label=name)
        axes[1].plot(log['t'], np.degrees(log['theta']), label=name)
        axes[2].plot(log['t'], log['force'], label=name)
    log = next(iter(logs.values()))
    axes[0].plot(log['t'], log['target'], 'k--', lw=1, label='setpoint')
    if log['disturbed'].any():
        t0, t1 = log['t'][log['disturbed']][[0, -1]]
        for ax in axes:
            ax.axvspan(t0, t1, color='orange', alpha=0.3)
    axes[0].set_ylabel('trolley x [m]'); axes[1].set_ylabel('sway [deg]')
    axes[2].set_ylabel('force [N]'); axes[2].set_xlabel('time [s]')
    axes[0].legend(fontsize=8); axes[0].set_title(title)
    plt.tight_layout()


def crane_report(log, dt=CranePlant.default_dt):
    m = step_response_metrics(torch.tensor(log['x']), target=5.0, dt=dt, initial=0.0,
                              control=torch.tensor(log['force'])).as_dict()
    m['max_sway_deg'] = float(np.degrees(np.abs(log['theta']).max()))
    return m


def show_table(columns):
    """Print {column: {metric: value}} as a small aligned table."""
    names = list(columns)
    rows = list(next(iter(columns.values())))
    print(f"{'':20s}" + ''.join(f'{n:>26s}' for n in names))
    for r in rows:
        print(f'{r:20s}' + ''.join(f'{float(columns[n][r]):26.3f}' for n in names))


base_log = crane_log(base_video)
plot_crane({'state feedback': base_log}, 'Crane step response - baseline')
show_table({'state feedback': crane_report(base_log)})

Controllers are compared with **classical metrics** (rise time, overshoot, settling
time, IAE/ISE/ITAE, control effort), not with the RL reward - that is what makes PID,
fuzzy and RL controllers comparable.

## 3. A hand-written fuzzy crane controller

Our own controller: two small Mamdani systems added together.

- **Position** FIS: inputs `x_error`, `x_error_dot` → force that drives the trolley.
- **Anti-sway** FIS: inputs `theta_error`, `theta_error_dot` → correction force.
  Note the sign: to damp a pendulum, the trolley must move *towards* the swing.

Both use 3 terms per input (`N`egative, `Z`ero, `P`ositive), 5 output terms and the
same 3×3 "PD-like" rule table. The universe of each input is `[-span, span]`.

In [ ]:
def nzp(name, span):
    v = FuzzyVariable(name, -span, span)
    v.add_term('N', TrapezoidalMF(-span, -span, -span / 2, 0))
    v.add_term('Z', TriangularMF(-span / 2, 0, span / 2))
    v.add_term('P', TrapezoidalMF(0, span / 2, span, span))
    return v

def output_var(name, span):
    v = FuzzyVariable(name, -span, span)
    for term, (a, b, c) in {'NB': (-span, -span, -span / 2), 'NS': (-span, -span / 2, 0),
                            'Z': (-span / 2, 0, span / 2),
                            'PS': (0, span / 2, span), 'PB': (span / 2, span, span)}.items():
        v.add_term(term, TriangularMF(a, b, c))
    return v

#            e_dot:  N     Z     P
PD_TABLE = {'N':   ['NB', 'NS', 'Z'],
            'Z':   ['NS', 'Z',  'PS'],
            'P':   ['Z',  'PS', 'PB']}

def pd_fis(e, de, out):
    rules = [FuzzyRule([(e, te), (de, td)], (out, PD_TABLE[te][j]))
             for te in 'NZP' for j, td in enumerate('NZP')]
    return FuzzyModule([e, de], out, rules, no_rule_value=0.0)

position_fis = pd_fis(nzp('x_error', 3.0), nzp('x_error_dot', 1.5), output_var('F_pos', 30.0))
sway_fis     = pd_fis(nzp('theta_error', 0.2), nzp('theta_error_dot', 0.6), output_var('F_sway', 30.0))
position_fis.print_rules()
position_fis.input_variables['x_error'].plot();

**Watch the universe!** The trolley starts 5 m from the target, but `x_error` only
spans ±3 m. Outside its universe every term has membership 0, so *no rule fires* and
the controller would output nothing. We therefore **saturate** each input to its
variable's range before evaluating the FIS - a standard trick in fuzzy control.

In [ ]:
def saturate(fis, values):
    """Clamp each crisp input to its variable's universe of discourse."""
    return {name: torch.tensor(float(v)).clamp(fis.input_variables[name].range_min,
                                               fis.input_variables[name].range_max)
            for name, v in values.items()}

class FuzzyCrane:
    """obs = [x_err, x_err_dot, theta_err, theta_err_dot] -> [force]."""
    def __init__(self, position_fis, sway_fis, sway_gain=-0.5, max_force=30.0):
        self.pos, self.sway = position_fis, sway_fis
        self.sway_gain, self.max_force = sway_gain, max_force

    def __call__(self, obs):
        ex, dex, et, det = obs
        with torch.no_grad():
            f = self.pos(saturate(self.pos, {'x_error': ex, 'x_error_dot': dex})) \
                + self.sway_gain * self.sway(saturate(self.sway, {'theta_error': et,
                                                                  'theta_error_dot': det}))
        return np.array([float(f.clamp(-self.max_force, self.max_force))], dtype=np.float32)

fuzzy_crane = FuzzyCrane(position_fis, sway_fis)
fuzzy_video = record(crane_env(), fuzzy_crane, every=5)
compare(**{'state feedback': base_video, 'fuzzy': fuzzy_video})

Same scenario, side by side. Now the numbers:

In [ ]:
fuzzy_log = crane_log(fuzzy_video)
plot_crane({'state feedback': base_log, 'fuzzy': fuzzy_log}, 'Crane step response - baseline vs fuzzy')
show_table({'state feedback': crane_report(base_log), 'fuzzy': crane_report(fuzzy_log)})

With just 2 × 9 readable rules, the fuzzy controller settles faster than the baseline
with roughly a third of the sway, at the price of a higher IAE: it accelerates more
gently, using about a quarter of the control effort.

Why does it behave like that? Ask the position FIS what it does at the start of the
move (5 m to go, saturated to 3 m, standing still):

In [ ]:
position_fis.plot_inference(saturate(position_fis, {'x_error': 5.0, 'x_error_dot': 0.0}));

In [ ]:
position_fis.plot_surface();

### Disturbance rejection

A second scenario holds the setpoint and hits the trolley with a 25 N impulse at
t = 10 s. Good controllers recover quickly without large sway.

In [ ]:
DIST = disturbance_rejection(1500, setpoint=[5.0, 0.0], time=10.0, magnitude=25.0, duration=0.2)
dist_videos = {'state feedback': record(crane_env(DIST), baseline, every=5),
               'fuzzy':          record(crane_env(DIST), fuzzy_crane, every=5)}
compare(**dist_videos)

In [ ]:
logs = {name: crane_log(v) for name, v in dist_videos.items()}
plot_crane(logs, 'Crane disturbance rejection (25 N impulse at t = 10 s)')
step0 = int(10.0 / CranePlant.default_dt)
show_table({name: disturbance_metrics(torch.tensor(log['x']), target=5.0,
                                      dt=CranePlant.default_dt, disturbance_step=step0)
            for name, log in logs.items()})

**Your turn:** change the `span`s, `sway_gain` or the rule table, re-run the cells
above and watch the effect. Which settings trade speed against sway?

## 4. Fuzzy Grand Prix

A dynamic kart races on a closed circuit. There are three circuits:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, name in zip(axes, ('oval', 'chicane', 'technical')):
    track = make_track(name)
    pts = np.vstack([track.points, track.points[:1]])
    ax.plot(pts[:, 0], pts[:, 1], color='#999', lw=14, solid_capstyle='round')
    ax.plot(pts[:, 0], pts[:, 1], 'w--', lw=1)
    ax.plot(*track.points[0], 'k>', ms=10)
    ax.set_aspect('equal'); ax.set_title(f'{name}  ({track.length:.0f} m)')
plt.tight_layout()

The observation is **track-relative**, so the same rule base works on every circuit:

`[cross_track_error, heading_error, speed, yaw_rate, lookahead_curvature, progress]`

The action is `[throttle, steering]` in `[-1, 1]`. A lap ends at the finish line;
leaving the track is a **DNF** (did not finish), so corners cannot be cut.

The library ships a reference `FuzzyRaceDriver`. One lap of the chicane:

In [ ]:
def race(driver, track='chicane', every=4):
    return record(RacingEnv(track, render_mode=RENDER_MODE, render_backend=BACKEND),
                  driver, every=every)

def lap_result(video):
    info = video.info
    return f"{info['lap_time']:.2f} s" if info['lap_time'] else 'DNF (off track)'

ref_video = race(FuzzyRaceDriver())
print('lap time:', lap_result(ref_video))
ref_video

Every circuit, drawn from the recorded positions (`video.states[:, :2]`) and coloured
by speed. Here we only need the trajectories, so `every=1000` skips almost all frames.

In [ ]:
def plot_race(videos, title):
    fig, axes = plt.subplots(1, len(videos), figsize=(5 * len(videos), 4.5), squeeze=False)
    for ax, (name, video) in zip(axes[0], videos.items()):
        track = make_track(name)
        pts = np.vstack([track.points, track.points[:1]])
        ax.plot(pts[:, 0], pts[:, 1], color='#ccc', lw=14, solid_capstyle='round')
        speed = [i['speed'] for i in video.infos]
        sc = ax.scatter(video.states[:, 0], video.states[:, 1], c=speed, s=3,
                        cmap='plasma', vmin=0, vmax=4)
        ax.set_title(f'{name}: {lap_result(video)}'); ax.set_aspect('equal')
    fig.colorbar(sc, ax=axes[0].tolist(), label='speed [m/s]')
    fig.suptitle(title)

ref = {t: race(FuzzyRaceDriver(), t, every=1000) for t in ('oval', 'chicane', 'technical')}
plot_race(ref, 'Reference FuzzyRaceDriver')

### Drivers as JSON

For the live class leaderboard (`examples/race_control/`) a driver is just a JSON file
with two fuzzy systems - `throttle` and `steering` - in the format from notebook 01.
Each system may read any of the observation names above (plus `curve`, the absolute
curvature). `ConfigRaceDriver` turns the JSON into a driver whose `.throttle` and
`.steering` are ordinary fuzzy systems - so all the plotting tools work on them.

In [ ]:
example_spec = json.loads((REPO / 'examples/race_control/example_driver.json').read_text())
print('systems:', list(example_spec))
print('steering inputs:', list(example_spec['steering']['inputs']))
print('throttle rules :', len(example_spec['throttle']['rules']))

json_driver = ConfigRaceDriver(example_spec)
json_driver.throttle.plot_surface();

In [ ]:
plot_race({t: race(json_driver, t, every=1000) for t in ('oval', 'chicane', 'technical')},
          'example_driver.json')

### Write your own driver

A minimal driver: steer against the heading and cross-track errors, and lift off the
throttle when the road ahead curves. Edit the terms and rules, then re-run the cell and
watch your lap.

In [ ]:
my_driver_spec = {
    "steering": {
        "inputs": {
            "heading_error": {"range": [-3.15, 3.15], "terms": {
                "left":     ["trap", -3.15, -3.15, -0.4, 0.0],
                "straight": ["tri", -0.4, 0.0, 0.4],
                "right":    ["trap", 0.0, 0.4, 3.15, 3.15]}},
            "cross_track_error": {"range": [-1.0, 1.0], "terms": {
                "left":   ["trap", -1, -1, -0.3, 0.0],
                "centre": ["tri", -0.3, 0.0, 0.3],
                "right":  ["trap", 0.0, 0.3, 1, 1]}}
        },
        "output": {"range": [-1, 1], "terms": {
            "hard_right": ["tri", -1, -1, -0.5], "right": ["tri", -1, -0.5, 0],
            "none": ["tri", -0.5, 0, 0.5],
            "left": ["tri", 0, 0.5, 1], "hard_left": ["tri", 0.5, 1, 1]}},
        "rules": [
            {"if": {"heading_error": "left"},  "then": "hard_right"},
            {"if": {"heading_error": "right"}, "then": "hard_left"},
            {"if": {"heading_error": "straight", "cross_track_error": "left"},   "then": "left"},
            {"if": {"heading_error": "straight", "cross_track_error": "centre"}, "then": "none"},
            {"if": {"heading_error": "straight", "cross_track_error": "right"},  "then": "right"}
        ]
    },
    "throttle": {
        "inputs": {"curve": {"range": [0, 1.5], "terms": {
            "straight": ["tri", 0, 0, 0.3],
            "bend":     ["trap", 0.1, 0.4, 1.5, 1.5]}}},
        "output": {"range": [0, 1], "terms": {
            "slow": ["tri", 0, 0.3, 0.6], "fast": ["tri", 0.5, 1, 1]}},
        "rules": [
            {"if": {"curve": "straight"}, "then": "fast"},
            {"if": {"curve": "bend"},     "then": "slow"}
        ]
    }
}

my_video = race(ConfigRaceDriver(my_driver_spec), 'chicane')
print('my driver   :', lap_result(my_video))
print('reference   :', lap_result(ref_video))
compare(**{'reference driver': ref_video, 'my driver': my_video})

> **Sign check.** If your kart spins out immediately, flip the steering signs:
> print a few observations and actions to see which way "positive" turns.
>
> To join the class leaderboard, save the spec with
> `json.dump(my_driver_spec, open('my_driver.json', 'w'), indent=2)` and submit it at
> the URL your instructor gives you (the server is `examples/race_control/server.py`).

## 5. Quadrotor

A 13-state rigid-body quadrotor: world position and velocity, a unit quaternion and
body angular rates. Two action modes are available:

- `ControlMode.MOTOR_THRUST`: four rotor thrusts [N] (the physical actuators),
- `ControlMode.RATE_THRUST`: collective thrust + roll / pitch / yaw rate requests
  (what a flight controller usually exposes - easier for fuzzy design).

First, open-loop hovering: each rotor carries a quarter of the weight, so the drone
should simply stay where it is.

In [ ]:
drone = QuadrotorPlant(mode=ControlMode.MOTOR_THRUST)
hover = np.array([float(drone.m * drone.g / 4)] * 4, dtype=np.float32)
print(f'mass {drone.m:.2f} kg -> hover thrust per rotor {hover[0]:.3f} N')
hover_video = record(FuzzyControlEnv(drone, episode_steps=150, render_mode=RENDER_MODE,
                                     render_backend=BACKEND),
                     lambda obs: hover, every=3)
print(f'altitude drift: {hover_video.states[-1, 2] - hover_video.states[0, 2]:+.2e} m')
hover_video

Now **closed-loop altitude control** with the `DroneStarter`: a factorised fuzzy
controller of four small TSK systems (`roll`, `pitch`, `vertical`, `yaw`) from the
student workspace (section 6). Its consequents start at **zero**, so out of the box it
only hovers.

We give the `vertical` system a first-order consequent in every rule,
`f = k_e · z_error + k_v · z_velocity` - a fuzzy PD altitude controller - and ask it to
climb from the ground to 1.8 m, with three different gain pairs.

In [ ]:
sys.path.insert(0, str(REPO / 'examples' / 'student'))
from fuzzy_starters import CraneStarter, RaceStarter, DroneStarter

def set_linear(system, gains):
    """Initialise every TSK rule of `system` as f = sum(gain_i * input_i).

    The consequents normalise each input onto [-1, 1], so each raw gain is multiplied
    by the half-range of its input variable.
    """
    halves = [(v.range_max - v.range_min) / 2 for v in system.input_variables.values()]
    coeffs = torch.tensor([0.0] + [g * h for g, h in zip(gains, halves)])
    with torch.no_grad():
        for rule in system.rules:
            rule.consequent_fn.coefficients.copy_(coeffs)

def fly(pilot, target=(0.0, 0.0, 1.8), seconds=8.0, every=4):
    drone = QuadrotorPlant(mode=ControlMode.RATE_THRUST, target=target)
    scenario = Scenario(name='climb', episode_steps=round(seconds / drone.default_dt),
                        setpoint=list(drone.target))
    env = FuzzyControlEnv(drone, scenario=scenario, render_mode=RENDER_MODE,
                          render_backend=BACKEND)
    # The DroneStarter needs the raw state *and* the target.
    policy = lambda obs: pilot.act(obs, env.setpoint().detach().cpu().numpy())
    return record(env, policy, seed=7, every=every)

flights = {}
for gains in [(0.0, 0.0), (1.0, 0.5), (3.0, 2.5)]:
    pilot = DroneStarter()
    set_linear(pilot.vertical, gains)
    pilot.eval()
    flights[f'k_e={gains[0]}, k_v={gains[1]}'] = fly(pilot)

compare(**flights)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for name, video in flights.items():
    t = [i['time'] for i in video.infos]
    axes[0].plot(t, video.states[:, 2], label=name)
    axes[1].plot(t, [i['control'][0] for i in video.infos], label=name)
axes[0].axhline(1.8, color='k', ls='--', lw=1, label='target')
axes[0].set(xlabel='time [s]', ylabel='altitude z [m]')
axes[1].set(xlabel='time [s]', ylabel='collective thrust [N]')
axes[0].legend(fontsize=8)
fig.suptitle('Quadrotor altitude with the DroneStarter vertical FIS')
plt.tight_layout()
for name, video in flights.items():
    print(f'{name}: final altitude error {abs(video.states[-1, 2] - 1.8):.4f} m')

**Why only altitude?** The lateral `roll`/`pitch` systems map position error and
velocity straight to an angular **rate**. Without attitude feedback the tilt keeps
integrating and the drone flips over - try `set_linear(pilot.pitch, (0.6, 0.8))` with a
target of `(1, 0, 1.8)` and watch it happen. Fixing this (for example by adding the tilt
angle as an input, like the cascaded controller in `examples/drone_demo.py`) is one of
the coursework tasks.

## 6. The student workspace

`examples/student/fuzzy_starters.py` is **the only file you edit** for the coursework.
The plants, rewards and scenarios are fixed so that everybody's results are
comparable. It provides three starters with fixed input/output contracts:

| case | inputs | action | report |
|---|---|---|---|
| Crane | `[x_error, x_error_dot, theta_error, theta_error_dot]` | trolley force [N] | IAE, residual sway, control effort |
| Kart | steering: `[cross_track_error, heading_error]`; speed: `[abs(curvature), speed]` | `[throttle, steering]` | lap time, or DNF |
| Drone | position/velocity errors, attitude and body rates | `[collective_thrust, roll_rate, pitch_rate, yaw_rate]` | final and integrated 3-D position error |

Each starter is a grid of first-order TSK rules with 3 Gaussian terms per input and
**zeroed consequents** - a safe neutral controller that hides no solution.

In [ ]:
for cls in (CraneStarter, RaceStarter, DroneStarter):
    c = cls()
    n_rules = sum(len(m.rules) for m in c.modules() if isinstance(m, FuzzyModule))
    n_params = sum(p.numel() for p in c.parameters())
    print(f'{cls.__name__:13s} rules={n_rules:3d}  trainable parameters={n_params}')

print('\nCraneStarter on a sample observation:', CraneStarter()(np.array([5.0, 0.0, 0.1, 0.0])))

### Training a starter from data

The starters are `nn.Module`s, so they train like the systems in notebook 01. As a
first exercise we **imitate** the state-feedback baseline: take the observations it
visited in its video (plus some random perturbations), ask it what it would do there,
and fit the 81-rule `CraneStarter` to those actions with a batched regression.
*(About 20-40 seconds on a laptop CPU.)*

In [ ]:
rng = np.random.default_rng(0)
visited = np.asarray(base_video.observations)
X = np.vstack([visited, visited + rng.normal(size=visited.shape) * [0.5, 0.3, 0.05, 0.2]])
Y = np.array([baseline(x)[0] for x in X])
X, Y = torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)
print('dataset:', tuple(X.shape))

torch.manual_seed(0)
student = CraneStarter()
student.train()
opt = torch.optim.Adam(student.parameters(), lr=0.05)
losses = []
for epoch in range(300):
    opt.zero_grad()
    loss = ((student(X) - Y) ** 2).mean()     # batched forward: (N, 4) -> (N,)
    loss.backward()
    opt.step()
    losses.append(loss.item())
print(f'MSE: {losses[0]:.2f} -> {losses[-1]:.2f}')

In [ ]:
student.eval()
student_video = record(crane_env(), student, every=5)
compare(**{'state feedback (teacher)': base_video, 'CraneStarter (imitation)': student_video})

In [ ]:
student_log = crane_log(student_video)
plot_crane({'state feedback': base_log, 'hand fuzzy': fuzzy_log,
            'CraneStarter (imitation)': student_log}, 'Crane: three controllers')
show_table({'state feedback': crane_report(base_log), 'hand fuzzy': crane_report(fuzzy_log),
            'CraneStarter (imitation)': crane_report(student_log)})

Imitation can only copy the teacher. To *improve* on it, optimise a closed-loop
cost such as IAE plus sway. Because the plants are differentiable torch code, you can
backpropagate that cost through a simulated trajectory (`fuzzy_torch.envs.rk4_step`
or `rollout`, on the batched plant). This is far more expensive than imitation, so
start with short horizons and small rule bases.

## 7. Saving videos, live windows and scripts

- `video.save('lap.gif')` (or `.webp`) writes any recording to a file - handy for
  reports and slides.
- On your own computer, set `RENDER_MODE = 'human'` in section 0 and re-run: every
  `record(...)` call then opens a live window instead of embedding a video.
- The same demos also exist as scripts. From the repository root:

```bash
pip install -e ".[viz,test]"

python examples/crane_demo.py --live              # interactive 3-D viewer
python examples/racing_demo.py --live --backend mujoco
python examples/drone_demo.py --live

python examples/race_control/server.py            # class leaderboard (see its README)
pytest                                            # the test-suite
```

## 8. Exercises

1. **Crane.** Tune the hand-written fuzzy crane to match the baseline's IAE while
   keeping its low sway. Report IAE, settling time, maximum sway and control effort,
   and save a video of your best run.
2. **Rule table.** Replace the 3×3 table with a 5×5 one (NB, NS, Z, PS, PB inputs).
   Is the extra complexity worth it?
3. **Grand Prix.** `my_driver_spec` finishes every circuit but is slower than the
   reference driver. Add `speed` as a throttle input so the kart brakes *before* corners
   and beat the reference lap times. Submit your driver to the leaderboard.
4. **Drone.** Make the `DroneStarter` fly to `(1.4, -1.0, 1.8)`. You will need
   attitude information in the lateral systems (see the note in section 5).
5. **Learning.** Fit the `RaceStarter` by imitating `FuzzyRaceDriver` on the oval, then
   test it on the `technical` circuit. Does imitation generalise to an unseen track?